<a href="https://colab.research.google.com/github/paulosantosps/Case_Rank/blob/main/relatorio_desempenho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RELATÓRIO DA CAMPANHA
MÉTRICAS E KPIs


## 1. Conexão e carga

In [1]:
!pip install -q sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 24.6 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine   # usado só para conectar ao banco

try:
    from google.colab import userdata
    url = userdata.get("DATABASE_URL")
except Exception:
    url = os.environ["DATABASE_URL"]
engine = create_engine(url)

usuarios = pd.read_sql("SELECT * FROM gold.dim_usuario", engine)
inst = pd.read_sql("SELECT * FROM gold.ft_instalacao", engine)
conv = pd.read_sql("""
    SELECT *
    FROM gold.ft_conversao
""", engine)

# Validação: uma linha por chave primária
assert usuarios["user_id"].is_unique and inst["pk_instalacao"].is_unique and conv["pk_conversao"].is_unique, \
    "Há chaves repetidas na gold. Revise o script (dim_usuario, ft_instalacao e ft_conversao)."

print(len(usuarios), "usuários |", len(inst), "instalações |", len(conv), "pedidos")

32762 usuários | 25894 instalações | 12256 pedidos


## 2. Preparação

In [3]:
# Plataforma por Usuário
plataforma = usuarios.set_index("user_id")["platform"].map({"ios": "iOS", "android": "Android"})
inst["plataforma"] = inst["fk_user"].map(plataforma)
conv["plataforma"] = conv["fk_user"].map(plataforma)

# Indicadores dos Pedidos
conv["valido"] = conv["validate_event"] == "OK"
conv["faturado"] = conv["faturada"] == "Yes"
conv["valido_faturado"] = conv["validate_client"] == "OK"
conv["receita"] = np.where(conv["valido_faturado"] & (conv["event_revenue_currency"] == "BRL"),
                           conv["event_revenue"], 0.0)

# Indicadores das Instalações:
  #quais deles geraram pedido
  #quais pedidos são válidos
  #quais foram faturados
inst["gerou_pedido"] = inst["pk_instalacao"].isin(conv["fk_instalacao"])
inst["gerou_pedido_valido_faturado"] = inst["pk_instalacao"].isin(
    conv.loc[conv["valido_faturado"], "fk_instalacao"])

## 3. KPIs da campanha

In [5]:
instalacoes = len(inst)
pedidos = len(conv)
validos = conv["valido"].sum()
faturados = conv["faturado"].sum()
validos_faturados = conv["valido_faturado"].sum()
receita = conv["receita"].sum()

kpis = pd.Series({
    "Instalações": instalacoes,
    "Pedidos": pedidos,
    "Pedidos válidos": validos,
    "Pedidos faturados (no CRM)": faturados,
    "Pedidos válidos e faturados": validos_faturados,
    "Taxa de validação (%)": round(validos / pedidos * 100, 1),
    "Taxa de faturamento das válidas (%)": round(validos_faturados / validos * 100, 1),
    "Conversão final (%)": round(validos_faturados / pedidos * 100, 1),
    "Receita faturada válida (R$)": round(receita, 2),
    "Ticket médio (R$)": round(receita / validos_faturados, 2),
    "Instalações que geraram pedido (%)": round(inst["gerou_pedido"].mean() * 100, 1),
    "Instalações com pedido válido e faturado (%)": round(inst["gerou_pedido_valido_faturado"].mean() * 100, 1),
}, dtype=object)

kpis.to_frame("Valor")

,Valor
Instalações,25894
Pedidos,12256
Pedidos válidos,2749
Pedidos faturados (no CRM),4021
Pedidos válidos e faturados,2458
Taxa de validação (%),22.4
Taxa de faturamento das válidas (%),89.4
Conversão final (%),20.1
Receita faturada válida (R$),911881.68
Ticket médio (R$),370.99


## 4. Por plataforma

In [6]:
por_plat = pd.DataFrame({
    "instalacoes": inst.groupby("plataforma").size(),
    "instalacoes_com_pedido": inst.groupby("plataforma")["gerou_pedido"].sum(),
    "pedidos": conv.groupby("plataforma").size(),
    "pedidos_validos": conv.groupby("plataforma")["valido"].sum(),
    "pedidos_validos_faturados": conv.groupby("plataforma")["valido_faturado"].sum(),
    "receita_faturada_valida": conv.groupby("plataforma")["receita"].sum().round(2),
})
por_plat["taxa_validacao_%"] = (por_plat["pedidos_validos"] / por_plat["pedidos"] * 100).round(1)
por_plat["ticket_medio"] = (por_plat["receita_faturada_valida"] / por_plat["pedidos_validos_faturados"]).round(2)
por_plat["instalacoes_com_pedido_%"] = (por_plat["instalacoes_com_pedido"] / por_plat["instalacoes"] * 100).round(1)
por_plat.astype(object).T

plataforma,Android,iOS
instalacoes,7756,18138
instalacoes_com_pedido,206,2110
pedidos,1777,10479
pedidos_validos,123,2626
pedidos_validos_faturados,107,2351
receita_faturada_valida,35365.75,876515.93
taxa_validacao_%,6.9,25.1
ticket_medio,330.52,372.83
instalacoes_com_pedido_%,2.7,11.6


## 5. Pedidos por categoria

In [7]:
categorias = conv["categoria_conversao"].value_counts().to_frame("pedidos")
categorias["%"] = (categorias["pedidos"] / pedidos * 100).round(1)
categorias

,pedidos,%
categoria_conversao,,
Inválida e não faturada,7944,64.8
Válida e faturada,2458,20.1
Inválida e faturada,1563,12.8
Válida e não faturada,291,2.4


## 6. Por que os pedidos são reprovados

Um pedido pode falhar em mais de uma regra, então os números não somam o total.

In [8]:
motivos = pd.Series({
    "Atribuição não primária": (~conv["is_primary_attribution"].astype(bool)).sum(),
    "Data de atribuição anterior a 08/05/2025": (conv["validacao_data_atribuicao"] == "No").sum(),
    "Fuso horário fora da janela": (conv["validacao_time_zone"] == "No").sum(),
    "Moeda divergente": (conv["currency_validation"] == "No").sum(),
    "Pedido fora do padrão ABR": (conv["validacao_axx"] == "No").sum(),
    "País divergente": (conv["validacao_geo"] == "No").sum(),
    "Artigo inválido": (conv["validacao_article"] == "No").sum(),
}).sort_values(ascending=False)

motivos = motivos.to_frame("pedidos")
motivos["% dos pedidos"] = (motivos["pedidos"] / pedidos * 100).round(1)
motivos

,pedidos,% dos pedidos
Atribuição não primária,9253,75.5
Data de atribuição anterior a 08/05/2025,1171,9.6
Fuso horário fora da janela,228,1.9
Moeda divergente,46,0.4
Pedido fora do padrão ABR,46,0.4
País divergente,45,0.4
Artigo inválido,0,0.0
